# A Simple RAG Demo Project

Before running the notebook, follow the setup instructions in [README.md](README.md).

In [1]:
import os
import sys
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer
from openai import AzureOpenAI
from dotenv import load_dotenv

In [2]:
import requests, os
from dotenv import load_dotenv
load_dotenv(override=True)

url = os.getenv("AZURE_OPENAI_ENDPOINT") + "openai/deployments?api-version=2024-02-01"
r = requests.get(url, headers={"api-key": os.getenv("AZURE_OPENAI_API_KEY")})
print(r.json())

{'error': {'code': '404', 'message': 'Resource not found'}}


In [3]:
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import userdata
    os.environ["AZURE_OPENAI_API_KEY"] = userdata.get("AZURE_OPENAI_API_KEY")
    os.environ["AZURE_OPENAI_API_VERSION"] = userdata.get("AZURE_OPENAI_API_VERSION")
    os.environ["AZURE_OPENAI_ENDPOINT"] = userdata.get("AZURE_OPENAI_ENDPOINT")
    os.environ["AZURE_OPENAI_CHAT_DEPLOYMENT"] = userdata.get("AZURE_OPENAI_CHAT_DEPLOYMENT")
else:
    load_dotenv()

client = AzureOpenAI(
    api_key=os.getenv("AZURE_OPENAI_API_KEY"),
    api_version=os.getenv("AZURE_OPENAI_API_VERSION"),
    azure_endpoint=os.getenv("AZURE_OPENAI_ENDPOINT")
)
chat_deployment = os.getenv("AZURE_OPENAI_CHAT_DEPLOYMENT")
print("API version:", os.getenv("AZURE_OPENAI_API_VERSION"))
print("Deployment:", chat_deployment)
embedding_model = SentenceTransformer("paraphrase-multilingual-mpnet-base-v2")

API version: 2025-01-01-preview
Deployment: gpt-4.1-mini


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [4]:
DATA_DIR = "./data"
texts = []
source_files = []
for filename in os.listdir(DATA_DIR):
    if filename.endswith(".txt"):
        path = os.path.join(DATA_DIR, filename)

        with open(path, "r", encoding="utf-8") as f:
            file_text = f.read()

        texts.append(file_text)
        source_files.append(filename)

text = "\n\n".join(texts)
chunks = [p.strip() for p in text.split("\n\n") if p.strip()]
if not chunks:
    raise ValueError("No text chunks found. Add .txt files to the ./data folder.")

print("Loaded files:")
for filename in source_files:
    print("-", filename)

print("\nChunks:")
for c in chunks:
    print("-", c)

Loaded files:
- knowledge.txt
- library.txt
- university_rules.txt

Chunks:
- Humans only use 2% of their brains.
The Great Wall of China is visible from the Moon.
Drinking eight liters of water a day is essential for survival.
Lightning never strikes the same place twice.
- Mount Everest is the tallest mountain on Earth. Its peak is 8848 meters above sea level and it is part of the Himalayas.
- The Amazon rainforest is the largest tropical rainforest in the world. It spans multiple countries in South America and contains immense biodiversity.
- The Pacific Ocean is the largest ocean on Earth. It covers more than 30% of the planet’s surface.
- The university library is open from Monday to Friday from 9:00 to 18:00.
- Students can borrow books for 14 days.
- Study rooms can be booked online through the university library system.
- If a book is returned late, a small fine may be applied.
- Students who have unpaid library fines are not allowed to borrow new books until the fine is cleare

In [5]:
def get_embedding(text):
    emb = embedding_model.encode(text)
    emb = emb / np.linalg.norm(emb)
    return emb

chunk_embeddings = np.array([get_embedding(chunk) for chunk in chunks]).astype("float32")
embedding_dim = chunk_embeddings.shape[1]
index = faiss.IndexFlatIP(embedding_dim)
index.reset()
index.add(chunk_embeddings)
print("FAISS index size:", index.ntotal)

FAISS index size: 18


In [6]:
question = "Is it true that we can see the Great Wall of China from the Moon?"
print("Question:", question)
question_embedding = np.array([get_embedding(question)]).astype("float32")

Question: Is it true that we can see the Great Wall of China from the Moon?


In [7]:
k = 3
distances, indices = index.search(question_embedding, k)
retrieved_chunks = [chunks[i] for i in indices[0]]
print("Retrieved context:")
for score, c in zip(distances[0], retrieved_chunks):
    print(f"[score={score:.4f}] - {c}")

Retrieved context:
[score=0.4714] - Humans only use 2% of their brains.
The Great Wall of China is visible from the Moon.
Drinking eight liters of water a day is essential for survival.
Lightning never strikes the same place twice.
[score=0.3445] - Mount Everest is the tallest mountain on Earth. Its peak is 8848 meters above sea level and it is part of the Himalayas.
[score=0.2869] - The Pacific Ocean is the largest ocean on Earth. It covers more than 30% of the planet’s surface.


In [8]:
context = "\n\n".join(retrieved_chunks)
prompt = f"""
# Instructions
You are a Retrieval-Augmented Generation assistant.
Answer the question using only the provided context.
Do not use outside knowledge.
If the answer is not present in the context, say: "There is no information available in the knowledge base."
Keep the answer short and clear.

# Context:
{context}

# Question:
{question}

# Answer:
"""

In [9]:
response = client.chat.completions.create(
    model=chat_deployment,
    messages=[{"role": "user", "content": prompt}],
    temperature=0
)
print("\nLLM Answer:")
print(response.choices[0].message.content)


LLM Answer:
No, it is not true. The context states that "The Great Wall of China is visible from the Moon" as a fact, but this is a common misconception. However, since the context presents it as true, based on the provided information, yes, it is true.
